# Lezione 6 — LLM (GPT, Llama) e il tuo primo chatbot con Hugging Face

## Obiettivi di apprendimento

- Spiegare cos'è un LLM (Large Language Model) in termini semplici.
- Conoscere, ad alto livello, le caratteristiche della famiglia Llama 2 e perché
  il codice di esempio del suo PDF ufficiale è fittizio.
- Distinguere un modello seq2seq (encoder-decoder) da un LLM causale
  instruction-tuned.
- Costruire ed eseguire due chatbot da terminale 100% gratuiti con Hugging
  Face Transformers: `facebook/blenderbot-400M-distill` e
  `HuggingFaceTB/SmolLM2-360M-Instruct`.

## Cosa costruiremo

Due chatbot da terminale che girano sulla CPU gratuita di Colab, senza alcuna
API key. Potrai parlare con entrambi e confrontare come rispondono.

## Nota veloce: cos'è un LLM?

Un LLM è un modello di deep learning addestrato con moltissimo testo, capace di
generare, tradurre, riassumere o analizzare il linguaggio (definizione dal glossario
ufficiale del corso). GPT (OpenAI) e Llama (Meta AI) sono due famiglie molto
conosciute di LLM, entrambe basate sull'architettura Transformer e sul
meccanismo di attenzione visto nella lezione precedente — ma su una scala
molto più grande (miliardi di parametri).

**Avviso importante**: il PDF ufficiale "Introduction to Meta Llama 2" include
un esempio di codice con `from meta_llama import MetaLlama2` che **non
esiste come pacchetto reale** — lo stesso PDF lo riconosce e rimanda a Hugging
Face Transformers come via reale di implementazione. È esattamente quello che
useremo in questa lezione. Maggiori dettagli in
[manual/09_llms_gpt_llama.md](../manual/09_llms_gpt_llama_IT.md).

## Nota veloce: scegliere un modello per il tuo chatbot

Secondo il materiale sorgente, la scelta del modello dipende dal compito:

- **Chatbot / generazione di testo**: modelli seq2seq leggeri come
  `facebook/blenderbot-400M-distill` (girano su CPU) o LLM causali come
  GPT/Llama/Mistral (più potenti, più pesanti).
- **Analisi del sentiment / NER**: BERT o RoBERTa.
- **Traduzione**: MarianMT, T5, mT5 o NLLB.

Bisogna anche tenere conto di: licenza d'uso, dimensione del modello, dati
di addestramento e requisiti di prestazioni. Oggi useremo due modelli
aperti e leggeri: uno seq2seq e uno LLM causale.

## Passo 0: installare le dipendenze

Usiamo le stesse versioni specificate dal materiale sorgente.

In [ ]:
!pip install transformers==4.41.2 torch==2.2.2 accelerate==0.30.1 numpy==1.26.4

print("==== Librerie installate =====")

## Parte 1: chatbot con un modello seq2seq — `facebook/blenderbot-400M-distill`

Questo modello è leggero, aperto e ottimizzato per il dialogo. Costruisci tu
stesso il prompt come testo (`"User: ...\nBot:"`) e lo passi al
modello a ogni turno, insieme alla cronologia recente.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/blenderbot-400M-distill"

# Viene scaricato la prima volta; poi usa la copia locale
model_bb = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer_bb = AutoTokenizer.from_pretrained(model_name)

print("Modello Blenderbot caricato.")

In [ ]:
print("Chatbot (Blenderbot) pronto. Scrivi 'exit' per uscire.\n")
conversation_history = []

while True:
    # Teniamo solo gli ultimi scambi
    conversation_history = conversation_history[-6:]
    history_string = "\n".join(conversation_history)

    input_text = input("> ")
    if input_text.lower() == "exit":
        break

    prompt = history_string + f"\nUser: {input_text}\nBot:"

    inputs = tokenizer_bb(prompt, return_tensors="pt", truncation=True, max_length=512)

    outputs = model_bb.generate(
        **inputs,
        max_new_tokens=60,
        no_repeat_ngram_size=3,
        repetition_penalty=1.3,
        do_sample=True,
        temperature=0.6,
        top_p=0.85
    )

    response = tokenizer_bb.decode(outputs[0], skip_special_tokens=True).strip()
    print("Bot:", response)

    conversation_history.append(f"User: {input_text}")
    conversation_history.append(f"Bot: {response}")

## Parte 2: chatbot con un LLM causale — `HuggingFaceTB/SmolLM2-360M-Instruct`

Questo modello è un LLM causale instruction-tuned: invece di costruire il prompt a
mano, usiamo una lista strutturata di messaggi con ruoli
(`system`/`user`/`assistant`) e lasciamo che
`tokenizer.apply_chat_template(...)` costruisca il formato esatto che il
modello si aspetta.

In [ ]:
from transformers import AutoModelForCausalLM
import torch
import warnings

warnings.filterwarnings("ignore")

model_name_llm = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer_llm = AutoTokenizer.from_pretrained(model_name_llm)
tokenizer_llm.pad_token = tokenizer_llm.unk_token

model_llm = AutoModelForCausalLM.from_pretrained(
    model_name_llm,
    device_map="cpu",
    torch_dtype=torch.float32
)

print("Modello SmolLM2 caricato.")

In [ ]:
# Il system prompt definisce la personalità e il comportamento del bot
messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Give short and concise answers in 2-3 lines."}
]

print("Chatbot (SmolLM2) pronto. Scrivi 'exit' per uscire.\n")
while True:
    user_input = input("> ")
    if user_input.lower() == "exit":
        break

    messages.append({"role": "user", "content": user_input})
    # Conserviamo il messaggio di sistema + gli ultimi 10 turni
    messages = [messages[0]] + messages[-10:]

    tokenized = tokenizer_llm.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        max_length=512
    )

    with torch.inference_mode():
        outputs = model_llm.generate(
            tokenized["input_ids"],
            attention_mask=tokenized["attention_mask"],
            max_new_tokens=60,
            temperature=0.5,
            top_p=0.8,
            do_sample=True,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer_llm.pad_token_id
        )

    response = tokenizer_llm.decode(
        outputs[0][tokenized["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    print(f"Bot: {response}\n")

    messages.append({"role": "assistant", "content": response})

## Come sarebbe con le API a pagamento del materiale ufficiale (riferimento, non eseguibile)

Il PDF di Llama 2 e diversi progetti del corso (Babel Fish, chat vocale)
usano LLM a pagamento tramite watsonx o l'API di OpenAI. Per esempio, così si
chiama un modello di watsonx nel progetto Babel Fish (codice reale, non
eseguibile qui perché richiede credenziali di IBM Cloud):

```python
# Così si fa con watsonx (materiale ufficiale di IBM) — richiede API key,
# NON eseguirlo qui. Usiamo SmolLM2/Blenderbot come alternativa gratuita.
def watsonx_process_message(user_message):
    prompt = f"""
    Translate the following English sentence into Spanish.
    Reply ONLY with the translation, no explanations, no formatting, no extra text.
    English: {user_message}
    Spanish:
    """
    response_text = model.generate_text(prompt=prompt)
    return response_text.strip()
```

In questa lezione usiamo sempre la via gratuita (Hugging Face) in modo che tu
possa eseguire tutto il codice senza costi né account a pagamento.

## Riepilogo della lezione

- Un LLM viene addestrato con enormi quantità di testo per generare, tradurre
  o riassumere il linguaggio; GPT e Llama sono due famiglie molto conosciute.
- Il codice di esempio del PDF di Llama2 (`meta_llama.MetaLlama2`) è
  fittizio — Hugging Face Transformers è la via reale e gratuita.
- `facebook/blenderbot-400M-distill` (seq2seq) richiede costruire il prompt
  a mano come testo semplice.
- `HuggingFaceTB/SmolLM2-360M-Instruct` (LLM causale) usa
  `apply_chat_template` con messaggi di ruolo `system`/`user`/`assistant`.
- Entrambi i modelli girano su CPU senza bisogno di API key.

## Per continuare

- Ripassa la teoria in
  [manual/09_llms_gpt_llama.md](../manual/09_llms_gpt_llama_IT.md) e
  [manual/10_chatbot_huggingface.md](../manual/10_chatbot_huggingface_IT.md).
- Metti in pratica quanto imparato in
  [clase06_practica_llms_chatbot_huggingface.ipynb](../notebooks_practica/clase06_practica_llms_chatbot_huggingface_IT.ipynb),
  dove cambierai il system prompt, la temperature e il top_p del
  chatbot per dargli una nuova "personalità".